### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="in_vehicle_coupon_recommendation",
    dataset_year="2017",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5GS4P",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/in_vehicle_coupon_recommendation/ && wget -P local-data-warehouse/in_vehicle_coupon_recommendation/ https://archive.ics.uci.edu/static/public/603/in+vehicle+coupon+recommendation.zip && unzip local-data-warehouse/in_vehicle_coupon_recommendation/in+vehicle+coupon+recommendation.zip -d local-data-warehouse/in_vehicle_coupon_recommendation/
""",
    # References
    academic_reference_bibtex="""@article{wang2017bayesian,
  title={A bayesian framework for learning rule sets for interpretable classification},
  author={Wang, Tong and Rudin, Cynthia and Doshi-Velez, Finale and Liu, Yimin and Klampfl, Erica and MacNeille, Perry},
  journal={Journal of Machine Learning Research},
  volume={18},
  number={70},
  pages={1--37},
  year={2017}
}
""",
    academic_reference_bibtex_key="wang2017bayesian",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We renamed the target feature and its value to be more descriptive.
- We removed text data from the "time" feature to make it numeric.
- We drop constant columns.
- We fixed a typo in the feature names.
- Anomaly: the data has many binned numeric features that are treated as categorical features with text describing the bins.
- Anomaly: the numeric features in the dataset are low-cardinality (<25)
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="AcceptCoupon",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="AcceptCoupon",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/in-vehicle-coupon-recommendation.csv")

target_feature = "AcceptCoupon"
df = df.rename(columns={"Y": target_feature})
df[target_feature] = df[target_feature].map({0: "No", 1: "Yes"})

# Transform time to 24-hour format to be fully numeric.
df["time"] = (
    df["time"]
    .apply(
        lambda x: int(x.replace("AM", ""))
        if "AM" in x
        else int(x.replace("PM", "")) + 12
    )
    .astype(int)
)

# Drop constant columns
df = df.drop(columns=["toCoupon_GEQ5min"])
df = df.rename(columns={"passanger": "passenger"})


cat_features = [
    "expiration",
    'destination',
    'passenger',
    'weather',
    'coupon',
    'gender',
    'age',
    'maritalStatus',
    'has_children',
    'education',
    'occupation',
    'income',
    'car',
    'Bar', 
    'CoffeeHouse',
    'CarryAway', 
    'RestaurantLessThan20', 
    'Restaurant20To50',
    'toCoupon_GEQ15min',
    'toCoupon_GEQ25min',
    'direction_same', 
    'direction_opp',
    'AcceptCoupon'
]
df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()